# McCann interpolation from a transport plan

This notebook generates `fig:kantorovich-plan-interpolation`.  For a coupling
\[
\pi=\sum_{i,j}P_{ij}\delta_{(x_i,y_j)},
\]
the displacement interpolation is obtained by pushing the plan through
\[
e_t(x,y)=(1-t)x+ty,\qquad \alpha_t=(e_t)_\#\pi.
\]
When the plan is not induced by a map, one source atom can create several moving atoms, one for each active coefficient `P_ij`.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from scipy.optimize import linear_sum_assignment

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    RED,
    VIOLET,
    box_axes,
    canonical_matching_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
np.random.seed(0)

POINT_SIZE = 10.5
POINT_SIZE_LARGE = 12.0


def active_pairs(P, threshold=1e-11):
    return [(i, j, float(P[i, j])) for i in range(P.shape[0]) for j in range(P.shape[1]) if P[i, j] > threshold]


def mass_sizes(weights, base=POINT_SIZE_LARGE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / max(weights.max(), 1e-15)
    return base * (0.58 + 1.25 * np.sqrt(weights))


def draw_clouds(ax, x, y, a=None, b=None, *, alpha=1.0, size=POINT_SIZE, zorder=4):
    sx = np.full(len(x), size) if a is None else mass_sizes(a, base=size)
    sy = np.full(len(y), size) if b is None else mass_sizes(b, base=size)
    ax.scatter(x[:, 0], x[:, 1], s=sx, marker="o", color=RED, alpha=alpha, edgecolor="none", zorder=zorder)
    ax.scatter(y[:, 0], y[:, 1], s=sy, marker="o", color=BLUE, alpha=alpha, edgecolor="none", zorder=zorder)


def set_cloud_limits(ax, x, y, pad=0.18):
    (xmin, xmax), (ymin, ymax) = padded_limits(np.vstack([x, y]), pad=pad)
    span = max(xmax - xmin, ymax - ymin)
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    ax.set_xlim(cx - span / 2, cx + span / 2)
    ax.set_ylim(cy - span / 2, cy + span / 2)
    ax.set_aspect("equal")
    remove_axes(ax)


def draw_straight_segments(ax, x, y, pairs, *, color=VIOLET, min_width=0.12, max_width=1.20, alpha=0.55, zorder=1):
    if not pairs:
        return
    masses = np.array([m for _, _, m in pairs], dtype=float)
    rel = masses / max(masses.max(), 1e-15)
    segments = [[x[i], y[j]] for i, j, _ in pairs]
    base = np.array(to_rgb(color))
    colors = [(*base, min(alpha * (0.22 + 0.78 * np.sqrt(r)), 0.9)) for r in rel]
    widths = min_width + (max_width - min_width) * np.sqrt(rel)
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=widths, zorder=zorder))


def draw_curved_segments(ax, x, y, pairs, *, color=VIOLET, min_width=0.10, max_width=1.20, alpha=0.55, curvature=0.11, zorder=1):
    if not pairs:
        return
    masses = np.array([m for _, _, m in pairs], dtype=float)
    rel = masses / max(masses.max(), 1e-15)
    base = np.array(to_rgb(color))
    curves, colors, widths = [], [], []
    for (i, j, _), r in zip(pairs, rel):
        p, q = x[i], y[j]
        mid = 0.5 * (p + q)
        direction = q - p
        normal = np.array([-direction[1], direction[0]])
        norm = np.linalg.norm(normal)
        if norm > 1e-12:
            normal = normal / norm
        sign = -1 if ((i + 2 * j) % 2) else 1
        control = mid + sign * curvature * normal * np.linalg.norm(direction)
        ts = np.linspace(0, 1, 16)
        curve = ((1 - ts)[:, None] ** 2) * p + (2 * (1 - ts) * ts)[:, None] * control + (ts[:, None] ** 2) * q
        curves.append(curve)
        colors.append((*base, min(alpha * (0.22 + 0.78 * np.sqrt(r)), 0.88)))
        widths.append(min_width + (max_width - min_width) * np.sqrt(r))
    ax.add_collection(LineCollection(curves, colors=colors, linewidths=widths, zorder=zorder))

## A sparse non-deterministic plan

The point locations again follow the canonical matching geometry, but the target has fewer weighted atoms than the source.  The optimal quadratic plan is sparse but not deterministic, which makes the branching structure of plan interpolation visible.

In [ ]:
fig_name = "kantorovich-plan-interpolation"
out = figure_dir(fig_name)

x, y, _ = canonical_matching_clouds(seed=2035, n_source=10, target_counts=(4, 3, 2))
a = np.ones(len(x)) / len(x)
raw = np.array([0.15, 0.08, 0.14, 0.11, 0.13, 0.09, 0.16, 0.10, 0.12])
b = raw / raw.sum()
C = ot.dist(x, y, metric="euclidean") ** 2
P = ot.emd(a, b, C, numItermax=200000)
support = active_pairs(P, threshold=1e-10)

## Time slices

Every panel shows the same faint input and target clouds, the active support of the plan as thin gray segments, and the moving atoms at time `t`.  Moving atom sizes encode the masses `P_ij` and their color interpolates from red to blue.

In [ ]:
def interpolated_support(t):
    pts, masses = [], []
    for i, j, mass in support:
        pts.append((1 - t) * x[i] + t * y[j])
        masses.append(mass)
    return np.asarray(pts), np.asarray(masses)


def draw_time_panel(t, path):
    fig, ax = plt.subplots(figsize=(2.05, 2.05))
    draw_straight_segments(ax, x, y, support, color=GRAY, min_width=0.10, max_width=0.34, alpha=0.34, zorder=1)
    draw_clouds(ax, x, y, a=a, b=b, alpha=0.30, size=POINT_SIZE, zorder=2)
    pts, masses = interpolated_support(t)
    ax.scatter(
        pts[:, 0], pts[:, 1],
        s=mass_sizes(masses, base=POINT_SIZE_LARGE),
        marker="o",
        color=interp_color(t),
        edgecolor="none",
        alpha=0.96,
        zorder=4,
    )
    set_cloud_limits(ax, x, y, pad=0.18)
    save_pdf(fig, path)
    plt.close(fig)


for t, filename in [(0.0, "time-000.pdf"), (0.25, "time-025.pdf"), (0.5, "time-050.pdf"), (0.75, "time-075.pdf"), (1.0, "time-100.pdf")]:
    draw_time_panel(t, out / filename)